# EX10: Implement PageRank 

## Overview
In this notebook, you will implement the **PageRank** algorithm.

You will break the work into small steps:
1. Load a graph
2. Construct the transition model
3. Implement PageRank via power iteration
4. Validate with sanity checks
5. Analyze results (top nodes, distribution, sensitivity)

You should have a more intuitive understanding of PageRank after completing the assignment.

You may use `numpy` and `matplotlib` for computations and plots.

**Do not** use `networkx.pagerank` or any other built-in PageRank implementation.


In [1]:
import numpy as np 
import networkx as nx 
import pylab as plot
import pandas as pd

## 1. Load the graph 


Included in the repository is a file `epl_final.csv`. This is a record of all matches in the English Premier League from 2000 - 2025. 

Load the csv with Pandas and transform the data into a graph. You should 
- Filter the dataframe to only matches that occured in the 2024/2025 season. 
- Add a vertex for every unique team
- Add a directed edge A -> B if team B won against team A. 

In [ ]:
df = pd.read_csv("epl_final.csv")
df = df[df["Season"] == "2024/25"]

G = nx.DiGraph()
G.add_nodes_from(pd.unique(df[["HomeTeam", "AwayTeam"]].values.ravel()))

for _, row in df.iterrows():
    if row["FullTimeResult"] == "H":
        G.add_edge(row["AwayTeam"], row["HomeTeam"])
    elif row["FullTimeResult"] == "A":
        G.add_edge(row["HomeTeam"], row["AwayTeam"])

## 2. Build the transition matrix

In [ ]:
def buildTransitionMatrix(G):
    n = G.number_of_nodes()
    nodes = list(G.nodes())
    idx = {v: i for i, v in enumerate(nodes)}
    P = np.zeros((n, n), dtype=np.float32)
    for i, v in enumerate(nodes):
        out = list(G.successors(v))
        if len(out) == 0:
            P[i, :] = 1 / n
        else:
            for u in out:
                P[i, idx[u]] = 1 / len(out)
    return P

## 2.5 Full transition matrix with teleportation

In [ ]:
def buildFullMatrix(G, d=0.85):
    n = G.number_of_nodes()
    P = buildTransitionMatrix(G)
    return d * P + (1 - d) / n

## 3. PageRank (dense)

In [ ]:
def PageRank(G, d=0.85, tol=1e-6, max_iter=200):
    n = G.number_of_nodes()
    r = (1 / n) * np.ones(n, dtype=np.float32)
    P = buildFullMatrix(G, d)
    for _ in range(max_iter):
        r_new = P.T @ r
        if np.sum(np.abs(r_new - r)) <= tol:
            r = r_new
            break
        r = r_new
    return r

r = PageRank(G)
print(f"Sum of entries is {np.sum(r)}, should be close to 1.")

## Sparse vs. Full 
You should conduct the below results on the `Wiki-Vote.txt` data (edgelist). This data includes a directed edge A -> B if user A voted to promote user B to an admin role on Wikipedia.

You can verify that such a large graph will be very slow (or even run out of memory) on the full dense matrix implementation.

## 5. Results: top nodes by PageRank

### Task
- Print the top-10 nodes by PageRank score
- Include their scores

Write a short 2-3 sentences on how these scores can be interpreted in your data.


**your interpretation**

## 6. Light analysis

### 6.1 Distribution of PageRank scores
Plot a histogram of PageRank scores.

Questions to answer:
- Is the distribution highly skewed?
- Do most nodes have very small score?


### 6.2 Compare damping factors

Run PageRank with different damping factors and compare:
- `d = 0.70`, `0.85`, `0.95`

Tasks:
1. Compute rank vectors for each `d`
2. Compare the **top-10 overlap** between settings
3. Compute L1 distances between the resulting rank vectors

Answer briefly:
- Which nodes are stable across damping factors?
- Does higher `d` concentrate more probability on a smaller set of nodes?


### 6.3 Relationship with out-degree

Compute correlation between PageRank and out-degree.

Questions:
- Are high PageRank nodes necessarily high out-degree nodes?
- What do you observe in the scatter plot?
